### 1. Исходные данные и расчет выборочных средних

Нам дано $n = 16$ пар наблюдений и следующие суммы:
* $\sum X = 96$
* $\sum Y = 64$
* $\sum X^2 = 657$
* $\sum Y^2 = 526$
* $\sum XY = 492$

Для начала найдем выборочные средние:
$$\bar{X} = \frac{\sum X}{n} = \frac{96}{16} = 6$$
$$\bar{Y} = \frac{\sum Y}{n} = \frac{64}{16} = 4$$

Далее вычислим суммы квадратов отклонений (девиации):
* **Для X ($SS_{XX}$):** $$SS_{XX} = \sum X^2 - n\bar{X}^2 = 657 - 16 \cdot 6^2 = 657 - 576 = 81$$
* **Для Y ($SS_{YY}$):** $$SS_{YY} = \sum Y^2 - n\bar{Y}^2 = 526 - 16 \cdot 4^2 = 526 - 256 = 270$$
* **Сумма произведений отклонений ($SS_{XY}$):**
  $$SS_{XY} = \sum XY - n\bar{X}\bar{Y} = 492 - 16 \cdot 6 \cdot 4 = 492 - 384 = 108$$

In [1]:
import numpy as np
import scipy.stats as stats

# Исходные данные
n = 16
sum_x = 96
sum_y = 64
sum_x2 = 657
sum_y2 = 526
sum_xy = 492

# Средние
mean_x = sum_x / n
mean_y = sum_y / n

# Девиации (суммы квадратов)
ss_xx = sum_x2 - n * (mean_x ** 2)
ss_yy = sum_y2 - n * (mean_y ** 2)
ss_xy = sum_xy - n * mean_x * mean_y

print(f"Средние: X = {mean_x}, Y = {mean_y}")
print(f"SS_xx = {ss_xx}, SS_yy = {ss_yy}, SS_xy = {ss_xy}")

Средние: X = 6.0, Y = 4.0
SS_xx = 81.0, SS_yy = 270.0, SS_xy = 108.0


### 2. Оценка параметров регрессии

Уравнение линейной регрессии имеет вид $y_i = a + bx_i$.
Оценки параметров по методу наименьших квадратов вычисляются следующим образом:

1. **Коэффициент наклона $b$ (slope):**
   $$b = \frac{SS_{XY}}{SS_{XX}} = \frac{108}{81} = \frac{4}{3} \approx 1.333$$

2. **Свободный член $a$ (intercept):**
   $$a = \bar{Y} - b\bar{X} = 4 - \left(\frac{4}{3}\right) \cdot 6 = 4 - 8 = -4$$

Таким образом, оцененное уравнение регрессии: 
$$\hat{y}_i = -4 + 1.333x_i$$

In [2]:
# Оценка коэффициентов
b = ss_xy / ss_xx
a = mean_y - b * mean_x

print(f"Коэффициент b (наклон): {b:.4f}")
print(f"Коэффициент a (пересечение): {a:.4f}")
print(f"Уравнение регрессии: y = {a:.4f} + {b:.4f} * x")

Коэффициент b (наклон): 1.3333
Коэффициент a (пересечение): -4.0000
Уравнение регрессии: y = -4.0000 + 1.3333 * x


### 3. Проверка гипотезы о коэффициенте $b$

Необходимо проверить гипотезу о том, что истинный коэффициент $b$ равен 1.
* **Нулевая гипотеза ($H_0$):** $b_{true} = 1$
* **Альтернативная гипотеза ($H_1$):** $b_{true} \neq 1$

Для проверки используется t-статистика Стьюдента:
$$t = \frac{b - 1}{S_b}$$

Где $S_b$ — стандартная ошибка коэффициента наклона. Чтобы ее найти, сначала вычислим остаточную сумму квадратов ($RSS$) и дисперсию остатков ($S^2$):
1. **Остаточная сумма квадратов ($RSS$):**
   $$RSS = SS_{YY} - b \cdot SS_{XY} = 270 - \left(\frac{4}{3}\right) \cdot 108 = 270 - 144 = 126$$
2. **Дисперсия остатков ($S^2$):**
   $$S^2 = \frac{RSS}{n - 2} = \frac{126}{14} = 9$$
3. **Стандартная ошибка $b$ ($S_b$):**
   $$S_b = \sqrt{\frac{S^2}{SS_{XX}}} = \sqrt{\frac{9}{81}} = \sqrt{\frac{1}{9}} = \frac{1}{3}$$

Теперь найдем наблюдаемое значение t-статистики:
$$t_{obs} = \frac{4/3 - 1}{1/3} = \frac{1/3}{1/3} = 1$$

При уровне значимости $\alpha = 0.05$ и числе степеней свободы $df = n - 2 = 14$, критическое значение распределения Стьюдента (двустороннее) равно $t_{crit} \approx 2.145$.

In [3]:
# Расчет стандартной ошибки и t-статистики
rss = ss_yy - b * ss_xy
df = n - 2
s_squared = rss / df

s_b = np.sqrt(s_squared / ss_xx)

# Тестируем H0: b = 1
b_hypothetical = 1
t_obs = (b - b_hypothetical) / s_b

# Находим p-value и критическое значение (alpha = 0.05, двусторонний тест)
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df)
p_value = 2 * (1 - stats.t.cdf(abs(t_obs), df))

print(f"Остаточная сумма квадратов (RSS): {rss:.4f}")
print(f"Стандартная ошибка Sb: {s_b:.4f}")
print(f"Наблюдаемое значение t-статистики: {t_obs:.4f}")
print(f"Критическое значение t (alpha={alpha}): {t_crit:.4f}")
print(f"p-value: {p_value:.4f}")
print("-" * 40)

if abs(t_obs) > t_crit:
    print("Отвергаем H0: Коэффициент b статистически значимо отличается от 1.")
else:
    print("Не отвергаем H0: Нет оснований утверждать, что коэффициент b отличается от 1.")

Остаточная сумма квадратов (RSS): 126.0000
Стандартная ошибка Sb: 0.3333
Наблюдаемое значение t-статистики: 1.0000
Критическое значение t (alpha=0.05): 2.1448
p-value: 0.3343
----------------------------------------
Не отвергаем H0: Нет оснований утверждать, что коэффициент b отличается от 1.


### 4. Вывод
Поскольку $|t_{obs}| = 1 < t_{crit} = 2.145$ (и $p\text{-value} \approx 0.33 > 0.05$), мы **не отвергаем нулевую гипотезу**. Данные не противоречат предположению о том, что истинный коэффициент регрессии равен единице.